### RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom `RAGFusion` class.
The pipeline uses **LLM-generated sub-queries** to improve retrieval quality, then fuses the results using **Reciprocal Rank Fusion (RRF)**.

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation - build context from retrieved documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

In [1]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

C:\Users\Shivam Singh\AppData\Local\Temp\ipykernel_5892\1720317915.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
This makes semantic similarity search possible at query time.

In [5]:
embedding_model =  GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag"
)

print("Vector store created successfully.")

Vector store created successfully.


### Step 4 - Create the Retriever

We configure a similarity-search retriever with `k=3`, meaning it will return the 3 most relevant chunks for any given query.

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

### Step 5 - RAG Fusion

`RAGFusion.from_llm` wires up the LLM to generate multiple sub-queries from the original query.
Each sub-query is sent to the retriever independently, and the results are merged using **Reciprocal Rank Fusion (RRF)** - documents that rank highly across multiple sub-queries bubble to the top.

In [16]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,
    k=3
)

In [17]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that the query and the document

--- Document 2 ---
search. Hybrid search combines the recall strengths of keyword matching with the semantic understanding of
dense retrieval.
Retrieved chunks are ranked and selected based on their relevance scores. To further improve retrieval
quality, NotebookLM may employ re-ranking techniques such as cross-encoder models, which score each
query-chunk pair more accurately than the initial bi-encoder retrieval step. The final set of retrieved chunks

--- Document 3 ---
When a user uploads a document to Notebook

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

In [18]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that the query and the document

search. Hybrid search combines the recall strengths of keyword matching with the semantic understanding of
dense retrieval.
Retrieved chunks are ranked and selected based on their relevance scores. To further improve retrieval
quality, NotebookLM may employ re-ranking techniques such as cross-encoder models, which score each
query-chunk pair more accurately than the initial bi-encoder retrieval step. The final set of retrieved chunks

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extr

### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

In [13]:
query

'How does NotebookLM retrieve relevant information from uploaded documents?'

In [19]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

When a user submits a query in NotebookLM, the system converts the query into an embedding. NotebookLM then uses a hybrid search approach, combining keyword matching with dense retrieval (semantic understanding), to search across the document collection. This involves using a vector index that supports efficient nearest-neighbor search. Retrieved chunks are ranked and selected based on their relevance scores, and re-ranking techniques, such as cross-encoder models, may be employed to further improve retrieval quality.
